# nb23b - Unrotated z < 0.35 map vs catalogue 1-halo sums

Companion to `23_unrotated_z0p35_tsz_ps.ipynb`. Loads the **cached map bandpowers**
from `data/map/nb23_datapoints/` (no map read, no NaMaster) and compares them to
two catalogue Poisson sums over **z < 0.35** halos (measured `Y_5R500c` amplitudes,
fixed A10 GNFW shape with **B = 1**):

1. **Solid** — full 1-halo sum $C_\ell^{1h}=(4\pi)^{-1}\sum_i |y_\ell(M_i,z_i)|^2$ with
   $y_\ell=Y_{\rm ang}\,\hat G(\ell\theta_{500})$ (GNFW form factor).
2. **Dotted** — white plateau $\ell\to0$: $C_\ell\to(4\pi)^{-1}\sum_i Y_{{\rm ang},i}^2$.

Masked cuts sum survivors $q<q_{\rm cut}$, matching map masks that remove $q>q_{\rm cut}$
clusters.

Run `23_unrotated_z0p35_tsz_ps.ipynb` once to produce the datapoint cache if missing.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Publication-quality plot defaults
plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "axes.linewidth": 0.8,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
})

from matplotlib.lines import Line2D

Z_MAX_MAP = 0.35
DP_NPZ = "/scratch/scratch-lxu/flamingo_repo/data/map/nb23_datapoints/nb23_unrot_z0p35_tsz_ps.npz"
CAT = "/scratch/scratch-lxu/flamingo_repo/data/hydro_L2p8m9/catalogue/halo_catalogue_M500c_5e13_zlt3_y0q_arnaudB1.csv"
OUTDIR = "/scratch/scratch-lxu/flamingo_repo/figures/nb23b_unrotated_z0p35_white_plateau"

CUTS = [
    ("fullsky", "Full sky", 1.0e9),
    ("qgt5", "q > 5", 5.0),
    ("qgt10", "q > 10", 10.0),
    ("qgt20", "q > 20", 20.0),
    ("qgt50", "q > 50", 50.0),
]


## Load cached map datapoints (nb23)

In [2]:
if not os.path.exists(DP_NPZ):
    raise FileNotFoundError(
        f"missing {DP_NPZ} — run 23_unrotated_z0p35_tsz_ps.ipynb first")

d = np.load(DP_NPZ)
ELL_EFF = d["ell_eff"]
measured = {}
for tag, lab, _ in CUTS:
    measured[tag] = dict(
        Dl=d[f"{tag}_Dl_map"],
        err=d[f"{tag}_sigma"],
        fsky_eff=float(d[f"{tag}_fsky_eff"]),
    )
    print(f"{lab:8s}  loaded  f_sky_eff={measured[tag]['fsky_eff']:.4f}")

Full sky  loaded  f_sky_eff=1.0000
q > 5     loaded  f_sky_eff=0.8021
q > 10    loaded  f_sky_eff=0.9138
q > 20    loaded  f_sky_eff=0.9790
q > 50    loaded  f_sky_eff=0.9988


## Catalogue sums (z < 0.35): GNFW $y_\ell$ and white plateau

In [3]:
from scipy.integrate import quad

ARCMIN_PER_RAD = 180.0 / np.pi * 60.0

df = pd.read_csv(CAT, comment="#")
df = df[(df["Y_5R500c_Mpc2"] > 0) & (df["R_500c_Mpc"] > 0) & np.isfinite(df["q"])]
df = df[df["z"] < Z_MAX_MAP]
R500 = df["R_500c_Mpc"].values
Ympc2 = df["Y_5R500c_Mpc2"].values
qv = df["q"].values
th500 = df["theta500_arcmin"].values / ARCMIN_PER_RAD
Yang = Ympc2 * (th500 / R500) ** 2                     # Y_ang = Y_5R500c / D_A^2  [sr]
print(f"catalogue clusters z<{Z_MAX_MAP}: {len(df)}")
print(f"white plateau  C_white = sum(Y_ang^2)/4pi = {np.sum(Yang**2)/(4*np.pi):.3e}  [sr]")

# Universal GNFW form factor (fixed A10, B=1) — same as nb14 / nb23
try:
    from hmfast.halos.profiles import GNFWPressureProfile
    prof = GNFWPressureProfile(B=1.0)
    P0, c500, al, be, ga = prof.P0, prof.c500, prof.alpha, prof.beta, prof.gamma
except ImportError:
  # fallback A10 numbers if hmfast not installed
    P0, c500, al, be, ga = 8.13, 1.156, 1.062, 5.4807, 0.3292

def p_shape(x):
    cx = c500 * np.maximum(x, 1e-12)
    return cx ** (-ga) * (1 + cx ** al) ** ((ga - be) / al)

I5 = quad(lambda x: p_shape(x) * x * x, 0, 5, limit=400)[0]
u_tab = np.geomspace(1e-3, 400.0, 600)
G_tab = np.array([quad(lambda x: p_shape(x) * x * x * np.sinc(u * x / np.pi), 0, 5, limit=400)[0] / I5
                  for u in u_tab])

def Ghat(u):
    return np.interp(np.clip(u, u_tab[0], u_tab[-1]), u_tab, G_tab)

def cl_1h(ell_arr, mask=None):
    w2 = (Yang if mask is None else Yang[mask]) ** 2
    t = th500 if mask is None else th500[mask]
    return np.array([np.sum(w2 * Ghat(l * t) ** 2) / (4 * np.pi) for l in ell_arr])

def c_white(mask=None):
    w = Yang if mask is None else Yang[mask]
    return np.sum(w ** 2) / (4 * np.pi)

def to_dl(ell_arr, cl):
    return ell_arr * (ell_arr + 1) * cl / (2 * np.pi)

ell_fine = np.geomspace(5.0, max(ELL_EFF.max(), 1200.0), 60)
white, cat_sum = {}, {}
for tag, lab, q_cat in CUTS:
    m = qv < q_cat if tag != "fullsky" else None
    cw = c_white(mask=m)
    white[tag] = dict(C=cw, Dl_fine=to_dl(ell_fine, cw), Dl_bin=to_dl(ELL_EFF, cw))
    cat_sum[tag] = dict(
        Dl_fine=to_dl(ell_fine, cl_1h(ell_fine, mask=m)),
        Dl_bin=to_dl(ELL_EFF, cl_1h(ELL_EFF, mask=m)),
    )
    if tag != "fullsky":
        print(f"{lab:8s}  survivors q<{q_cat:.0f}: {int((qv < q_cat).sum()):>6d}  "
              f"C_white={cw:.3e} sr")

catalogue clusters z<0.35: 133532
white plateau  C_white = sum(Y_ang^2)/4pi = 2.906e-16  [sr]


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


q > 5     survivors q<5: 131560  C_white=9.767e-18 sr


q > 10    survivors q<10: 133045  C_white=6.746e-17 sr


q > 20    survivors q<20: 133441  C_white=2.267e-16 sr


q > 50    survivors q<50: 133530  C_white=2.844e-16 sr


## Figure: map vs white plateau

In [4]:
sel = (ELL_EFF > 100) & (ELL_EFF < 900)
colors = plt.cm.plasma(np.linspace(0.0, 0.85, len(CUTS)))
fig, (ax, axr) = plt.subplots(2, 1, figsize=(7.4, 7.2), sharex=True,
                              gridspec_kw=dict(height_ratios=[3, 1], hspace=0.07))

for (tag, lab, _), col in zip(CUTS, colors):
    Dl_o = measured[tag]["Dl"]
    err = measured[tag]["err"]
    Dl_cat = cat_sum[tag]["Dl_bin"]
    ax.plot(ell_fine, cat_sum[tag]["Dl_fine"], "-", color=col, lw=2.2, zorder=3)
    ax.plot(ell_fine, white[tag]["Dl_fine"], ":", color=col, lw=1.2, zorder=1)
    ax.errorbar(ELL_EFF, Dl_o, yerr=err, fmt="o", color=col, ms=4, capsize=2,
                elinewidth=1, mfc="white", zorder=4)
    ratio = Dl_o / Dl_cat
    axr.errorbar(ELL_EFF, ratio, yerr=err / Dl_cat, fmt="o", color=col, ms=3,
                 capsize=2, elinewidth=1, mfc="white")
    print(f"{lab:8s}  median(map/cat-sum), 100<ell<900 = {np.median(ratio[sel]):.3f}  "
          f"median(map/white) = {np.median((Dl_o / white[tag]['Dl_bin'])[sel]):.3f}")

ax.set_xscale("log"); ax.set_yscale("log")
ax.set_ylabel(r"$D_\ell^{yy} = \ell(\ell+1)C_\ell^{yy}/2\pi$")
ax.set_title(r"Unrotated $z<0.35$: map vs catalogue 1-halo sum (GNFW $B=1$, measured $Y_{5R500c}$)")
ax.grid(True, which="both", alpha=0.2)
handles = [Line2D([0], [0], color=c, marker="o", mfc="white", lw=2, label=lab)
           for (_, lab, _), c in zip(CUTS, colors)]
handles += [
    Line2D([0], [0], color="k", lw=2.2, label="catalogue sum (GNFW B=1, measured Y)"),
    Line2D([0], [0], color="k", ls=":", lw=1.2,
           label=r"white plateau ($\sum Y_{\rm ang}^2/4\pi$)"),
    Line2D([0], [0], color="k", marker="o", mfc="white", ls="none", label="map (nb23 cache)"),
]
ax.legend(handles=handles, fontsize=7.5, loc="upper left", framealpha=0.9)

axr.axhline(1.0, color="k", lw=0.8, ls="--")
axr.fill_between([ELL_EFF.min(), ELL_EFF.max()], 0.9, 1.1, color="0.85", alpha=0.6)
axr.set_ylabel("map / cat sum"); axr.set_xlabel(r"multipole $\ell$")
axr.grid(True, which="both", alpha=0.2)

os.makedirs(OUTDIR, exist_ok=True)
fig.savefig(os.path.join(OUTDIR, "nb23b_white_plateau.png"), dpi=300, bbox_inches="tight")
fig.savefig(os.path.join(OUTDIR, "nb23b_white_plateau.pdf"), bbox_inches="tight")
print("saved ->", OUTDIR)
plt.show()

Full sky  median(map/cat-sum), 100<ell<900 = 0.953  median(map/white) = 0.080
q > 5     median(map/cat-sum), 100<ell<900 = 1.116  median(map/white) = 0.543
q > 10    median(map/cat-sum), 100<ell<900 = 1.020  median(map/white) = 0.136
q > 20    median(map/cat-sum), 100<ell<900 = 0.948  median(map/white) = 0.063
q > 50    median(map/cat-sum), 100<ell<900 = 0.924  median(map/white) = 0.076


saved -> /scratch/scratch-lxu/flamingo_repo/figures/nb23b_unrotated_z0p35_white_plateau


## Notes

- **Dashed** catalogue sum: $C_\ell^{1h}=(4\pi)^{-1}\sum_i Y_{{\rm ang},i}^2\hat G(\ell\theta_{500,i})^2$
  with measured `Y_5R500c` and fixed A10 GNFW shape ($B=1$), same as nb14 / nb23.
- **Dotted** white plateau: $\ell\to0$ limit $\hat G\to1$, so $C_\ell\to(4\pi)^{-1}\sum Y_{\rm ang}^2$
  (hence $D_\ell\propto\ell^2$).
- At low $\ell$ the map exceeds the 1-halo sums (2-halo clustering); at high $\ell$ the
  map / cat-sum ratio should approach unity.
- Datapoints and Knox errors come from `nb23_unrot_z0p35_tsz_ps.npz`; no HEALPix or NaMaster.